# 엑셀/PDF 파일 읽기 실습

- 엑셀 파일은 `openpyxl`, PDF 파일은 `pypdf` 패키지를 사용


## 패키지 설치

터미널에서 아래 명령어를 한 번 실행한다.

```bash
python -m pip install openpyxl pypdf
```


SyntaxError: invalid syntax (736633474.py, line 1)

## 엑셀 파일 읽기

엑셀 파일은 단순 문자열 파일이 아니라 여러 시트, 셀, 서식 정보가 들어 있는 구조화 파일이다. 그래서 `openpyxl` 같은 전용 라이브러리로 읽는다.


In [28]:
# load_workbook() : 엑셀을 열서서 Workbook 객체로 변환
from openpyxl import load_workbook

# data_only=True : 셀에 작성된 데이터 읽어오기(수식 X, 수식 결과 O)
workbook = load_workbook('students.xlsx', data_only=True)

print(workbook.sheetnames) # 엑셀의 모든 시트 읽어오기

worksheet = workbook['students']  # 'students' 시트만 얻어오기

print(worksheet.title) # 시트 이름('students')
print('worksheet.max_row: ', worksheet.max_row)
print('worksheet.max_column: ', worksheet.max_column)

['students']
students
worksheet.max_row:  5
worksheet.max_column:  5


In [29]:
# 특정 셀 값 읽어오기
print('B2 값:', worksheet['B2'].value)
print('D3 값:', worksheet['D3'].value)
print('4행 2열 값:', worksheet.cell(row=4, column=2).value)

B2 값: 홍길동
D3 값: 85
4행 2열 값: 유관순


In [30]:
# 엑셀 한 줄 씩 읽기
# iter == iterator == 반복자
# iter_rows == 행 반복자: 반복 될 때 마다 다음 행 반환
# values_only=True : 셀 안의 실제 값만 tuple로 가져오기
for row in worksheet.iter_rows(values_only=True):
    print(row)

('id', 'name', 'course', 'score', 'passed')
(1, '홍길동', 'Python IO', 92, True)
(2, '이순신', 'Python IO', 85, True)
(3, '유관순', 'Python IO', 78, True)
(4, '신사임당', 'Python IO', 64, False)


In [31]:
# 엑셀 행 데이터를 dict list로 변환
rows = list( worksheet.iter_rows( values_only=True) )
print(rows)

# 0행: ('id', 'name', 'course', 'score', 'passed')
# 1~4행: (1, '홍길동', 'Python IO', 92, True)

headers = rows[0]

student_list: list[dict] = []

for row in rows[1:]: # 0번 == 헤더, 0번을 제외한 1번 인덱스부터 시작
    # 학생 dict 데이터 생성
    # zip(tuple1, tuple2) -> 두 튜플의 같은 인덱스 요소끼리 K:V 쌍으로 묶어서 dict 변환
    student = dict(zip(headers, row))
    student_list.append(dict(student))

for student in student_list:
    print(student)

[('id', 'name', 'course', 'score', 'passed'), (1, '홍길동', 'Python IO', 92, True), (2, '이순신', 'Python IO', 85, True), (3, '유관순', 'Python IO', 78, True), (4, '신사임당', 'Python IO', 64, False)]
{'id': 1, 'name': '홍길동', 'course': 'Python IO', 'score': 92, 'passed': True}
{'id': 2, 'name': '이순신', 'course': 'Python IO', 'score': 85, 'passed': True}
{'id': 3, 'name': '유관순', 'course': 'Python IO', 'score': 78, 'passed': True}
{'id': 4, 'name': '신사임당', 'course': 'Python IO', 'score': 64, 'passed': False}


In [32]:
# 엑셀에서 읽어온 학생 데이터로 점수 학생수, 합계, 평균, 최고점, 최저점 구하기
# student_list를 사용


print("학생 수 :")
print("점수 합계: ")
print("점수 평균: ")
print("점수 최고점: ")
print("점수 최저점: ")

학생 수 :
점수 합계: 
점수 평균: 
점수 최고점: 
점수 최저점: 


In [33]:
# scores = [student['score'] for student in student_list]
scores: list[int] = [student['score'] for student in student_list]
if scores:
    student_count = len(scores) # 길이 (학생 수)
    total_score = sum(scores) # 합계
    avg_score = total_score / student_count # 평균
    max_score = max(scores) # 최고점
    min_score = min(scores) # 최저점

    print(f"학생 수 : {student_count}명")
    print(f"점수 합계: {total_score}점")
    print(f"점수 평균: {avg_score:.1f}점")
    print(f"점수 최고점: {max_score}점")
    print(f"점수 최저점: {min_score}점")

학생 수 : 4명
점수 합계: 319점
점수 평균: 79.8점
점수 최고점: 92점
점수 최저점: 64점


## PDF 파일 읽기

- PDF는 페이지, 글자 위치, 폰트, 이미지 정보가 함께 들어 있는 문서 파일이다. 그래서 텍스트 파일처럼 바로 읽기보다 `pypdf` 같은 라이브러리로 페이지 단위 텍스트를 추출한다.

- 한글 PDF는 폰트와 인코딩 방식에 따라 텍스트 추출 결과가 달라질 수 있음



In [36]:
# pypdf: 파이썬에서 pdf 파일을 읽고, 쓸 수 있게하는 라이브러리(모듈)
from pypdf import PdfReader

# pathlib.Patt : 파일 / 폴더 경로를 편하게 다루기 위한 클래스
from pathlib import Path
pdf_path = Path(r'C:\SKN_AI\02_python\_08_io\io_sample.pdf')
print(pdf_path)

reader = PdfReader(pdf_path)
# print(reader, type(reader))

print("reader.pages: ", reader.pages)

C:\SKN_AI\02_python\_08_io\io_sample.pdf
reader.pages:  [PageObject(0), PageObject(1)]


In [37]:
# 첫 번째 페이지 text 추출
first_page_text = reader.pages[0].extract_text()
print('첫 번째 페이지 text: ', first_page_text)

첫 번째 페이지 text:  IO Practice Report
This PDF is used for Python file I/O practice.
PDF files are structured documents, not plain text files.
We use pypdf to extract text page by page.



In [39]:
# 전체 페이지 text 추출
# 단, 페이지 별로 번호를 이용해서 구분

for page_number, page in enumerate(reader.pages, start=1):
    print(f"--- page_number: {page_number} ---")
    print(page.extract_text())

--- page_number: 1 ---
IO Practice Report
This PDF is used for Python file I/O practice.
PDF files are structured documents, not plain text files.
We use pypdf to extract text page by page.

--- page_number: 2 ---
Second Page
A PDF can contain multiple pages.
In Python, each page can be read and processed separately.

